In [38]:
import pandas as pd
import numpy as np
import re
from datetime import datetime, timedelta
import os

# Create directories
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/cleaned', exist_ok=True)
os.makedirs('outputs', exist_ok=True)

print("="*60)
print("DATA CLEANING UTILITY")
print("="*60)

# Generate a realistically messy dataset
np.random.seed(42)

# Customer data with intentional issues
customer_data = {
    'customer id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15],
    'Name': ['John Smith', 'JANE DOE', 'bob johnson', 'Alice Brown',
             'Charlie Wilson', '   Emma Davis   ', 'Michael Lee', 'SARA PARKER',
             'David Kim', 'Lisa Wong', 'Tom Harris', 'N/A', 'Unknown',
             'Rachel Green', 'Monica Geller'],
    'Email': ['john@email.com', 'jane.doe@domain.com', 'bob@',
              'alice@email', 'charlie@email.com', 'emma@email.com',
              'michael@email', 'sara@email.com', 'david@email.com',
              'lisa@email.com', 'tom@email.com', 'unknown@email.com',
              'NULL', 'rachel@email.com', 'monica@email.com'],
    'Phone': ['123-456-7890', '9876543210', '555-1234', '444-555-6666',
              '12345', '999-888-7777', '111-222-3333', '444-555-6666',
              '777-888-9999', '000-111-2222', '3334445555', 'N/A',
              'None', '555-666-7777', '888-999-0000'],
    'Age': [25, -5, 150, 30, 42, 18, 999, 35, 27, 'twenty', 38, 45, 22, 29, 31],
    'Registration_Date': ['2023-01-15', '2023/02/20', '15-03-2023', '2023.04.10',
                         '05/25/2023', '2023-06-30', '2023/07/04', '08-15-2023',
                         '2023.09.01', '10/10/2023', 'invalid date', '2023-12-01',
                         '2023-11-15', '2023-10-20', '2023-09-25'],
    'Total_Purchases': [100, 200, 300, None, 500, 600, 700, np.nan, 900,
                        1000, 1500, 0, -200, 800, 1200],
    'Membership_Status': ['Gold', 'Silver', 'Bronze', 'Gold', 'INVALID',
                          'Silver', 'Gold', 'Platinum', 'Bronze', 'Gold',
                          'Silver', 'N/A', 'Unknown', 'Gold', 'Silver'],
    'Last_Active': ['2024-01-01', '2024-01-15', '2024-01-10', '2024-01-20',
                    '2024-02-01', '2024-01-25', '2024-02-10', '2024-01-30',
                    '2024-02-05', '2024-01-18', '2024-02-12', '2024-01-22',
                    '2024-01-28', '2024-02-15', '2024-02-20']
}

df_messy = pd.DataFrame(customer_data)

# Add some duplicate rows
duplicate_row = df_messy.iloc[2].copy()
df_messy = pd.concat([df_messy, pd.DataFrame([duplicate_row])], ignore_index=True)
df_messy = pd.concat([df_messy, pd.DataFrame([duplicate_row])], ignore_index=True)

# Save messy dataset
df_messy.to_csv('data/raw/messy_customers.csv', index=False)
print("\n📊 Created messy dataset with:")
print(f"   - Rows: {len(df_messy)}")
print(f"   - Columns: {len(df_messy.columns)}")
print(f"   - Duplicates: 2 intentional duplicates")
print("\n📋 First look at messy data:")
print(df_messy.head(10))

DATA CLEANING UTILITY

📊 Created messy dataset with:
   - Rows: 17
   - Columns: 9
   - Duplicates: 2 intentional duplicates

📋 First look at messy data:
   customer id              Name                Email         Phone     Age  \
0            1        John Smith       john@email.com  123-456-7890      25   
1            2          JANE DOE  jane.doe@domain.com    9876543210      -5   
2            3       bob johnson                 bob@      555-1234     150   
3            4       Alice Brown          alice@email  444-555-6666      30   
4            5    Charlie Wilson    charlie@email.com         12345      42   
5            6     Emma Davis          emma@email.com  999-888-7777      18   
6            7       Michael Lee        michael@email  111-222-3333     999   
7            8       SARA PARKER       sara@email.com  444-555-6666      35   
8            9         David Kim      david@email.com  777-888-9999      27   
9           10         Lisa Wong       lisa@email.com  0

In [39]:
class DataCleaner:
    """Complete data cleaning utility class"""

    def __init__(self, df, name="dataset"):
        self.df = df.copy()
        self.name = name
        self.cleaning_log = []
        self.initial_shape = df.shape

    def log_action(self, action, details, before_count=None, after_count=None):
        """Log cleaning actions"""
        log_entry = {
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'action': action,
            'details': details,
            'before': before_count,
            'after': after_count
        }
        self.cleaning_log.append(log_entry)
        print(f"✓ {action}: {details}")

    def standardize_column_names(self):
        """Clean and standardize column names"""
        original_cols = self.df.columns.tolist()

        # Column name cleaning rules
        self.df.columns = (
            self.df.columns
            .str.lower()  # Convert to lowercase
            .str.strip()  # Remove leading/trailing spaces
            .str.replace(' ', '_')  # Replace spaces with underscores
            .str.replace('-', '_')  # Replace hyphens with underscores
            .str.replace('[^\w]', '_', regex=True)  # Replace special chars
            .str.replace('__+', '_', regex=True)  # Remove duplicate underscores
        )

        self.log_action(
            "Standardized column names",
            f"Renamed: {original_cols} → {self.df.columns.tolist()}"
        )
        return self

    def fix_data_types(self):
        """Detect and fix incorrect data types"""

        # 1. Fix numeric columns
        numeric_columns = ['age', 'total_purchases']
        for col in numeric_columns:
            if col in self.df.columns:
                before_invalid = self.df[col].isna().sum()

                # Convert to numeric, coerce errors to NaN
                self.df[col] = pd.to_numeric(self.df[col], errors='coerce')

                # Handle negative values in age
                if col == 'age':
                    invalid_neg = (self.df[col] < 0).sum()
                    self.df.loc[self.df[col] < 0, col] = np.nan
                    self.df.loc[self.df[col] > 120, col] = np.nan  # Cap age at 120

                # Handle negative purchases
                if col == 'total_purchases':
                    self.df.loc[self.df[col] < 0, col] = np.nan

                after_invalid = self.df[col].isna().sum()
                self.log_action(
                    "Fixed data types",
                    f"Column '{col}': converted {before_invalid} invalid values to NaN"
                )

        # 2. Fix dates
        if 'registration_date' in self.df.columns:
            before_invalid = self.df['registration_date'].isna().sum()

            # Try multiple date formats
            date_formats = ['%Y-%m-%d', '%Y/%m/%d', '%d-%m-%Y', '%Y.%m.%d', '%m/%d/%Y']

            for date_format in date_formats:
                mask = self.df['registration_date'].notna()
                self.df.loc[mask, 'registration_date'] = pd.to_datetime(
                    self.df.loc[mask, 'registration_date'],
                    format=date_format,
                    errors='ignore'
                )

            # Final conversion
            self.df['registration_date'] = pd.to_datetime(
                self.df['registration_date'],
                errors='coerce'
            )

            after_invalid = self.df['registration_date'].isna().sum()
            self.log_action(
                "Fixed date formats",
                f"Column 'registration_date': fixed {before_invalid} invalid dates"
            )

        # 3. Clean categorical columns
        if 'membership_status' in self.df.columns:
            valid_statuses = ['gold', 'silver', 'bronze', 'platinum']
            before_invalid = (~self.df['membership_status'].str.lower().isin(valid_statuses)).sum()

            self.df['membership_status'] = (
                self.df['membership_status']
                .str.lower()
                .str.strip()
            )
            self.df.loc[~self.df['membership_status'].isin(valid_statuses), 'membership_status'] = np.nan

            self.log_action(
                "Cleaned categorical data",
                f"Column 'membership_status': standardized {before_invalid} invalid entries"
            )

        return self

    def handle_missing_values(self, strategy='auto'):
        """Handle missing values with various strategies"""

        missing_before = self.df.isnull().sum().sum()

        # For numeric columns
        numeric_cols = self.df.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            if self.df[col].isnull().sum() > 0:
                if strategy == 'auto':
                    # Use median for skewed data, mean for normal
                    skewness = self.df[col].skew() if len(self.df[col].dropna()) > 1 else 0
                    if abs(skewness) > 1:
                        fill_value = self.df[col].median()
                        method = 'median'
                    else:
                        fill_value = self.df[col].mean()
                        method = 'mean'
                elif strategy == 'mean':
                    fill_value = self.df[col].mean()
                    method = 'mean'
                elif strategy == 'median':
                    fill_value = self.df[col].median()
                    method = 'median'
                elif strategy == 'mode':
                    fill_value = self.df[col].mode().iloc[0] if not self.df[col].mode().empty else 0
                    method = 'mode'
                else:
                    fill_value = 0
                    method = 'zero'

                self.df[col].fillna(fill_value, inplace=True)
                self.log_action(
                    "Handled missing values",
                    f"Column '{col}': filled {self.df[col].isnull().sum()} missing values with {method} ({fill_value:.2f})"
                )

        # For categorical columns
        categorical_cols = self.df.select_dtypes(include=['object']).columns
        for col in categorical_cols:
            if self.df[col].isnull().sum() > 0:
                mode_value = self.df[col].mode().iloc[0] if not self.df[col].mode().empty else 'unknown'
                self.df[col].fillna(mode_value, inplace=True)
                self.log_action(
                    "Handled missing values",
                    f"Column '{col}': filled missing values with mode '{mode_value}'"
                )

        missing_after = self.df.isnull().sum().sum()
        self.log_action(
            "Missing values summary",
            f"Reduced missing values from {missing_before} to {missing_after}",
            missing_before,
            missing_after
        )

        return self

    def remove_duplicates(self, subset=None, keep='first'):
        """Remove duplicate rows"""
        before_count = len(self.df)

        if subset is None:
            # Check all columns
            self.df = self.df.drop_duplicates(keep=keep)
            duplicate_cols = "all columns"
        else:
            self.df = self.df.drop_duplicates(subset=subset, keep=keep)
            duplicate_cols = f"columns: {subset}"

        after_count = len(self.df)
        duplicates_removed = before_count - after_count

        if duplicates_removed > 0:
            self.log_action(
                "Removed duplicates",
                f"Removed {duplicates_removed} duplicate rows based on {duplicate_cols}",
                before_count,
                after_count
            )
        else:
            self.log_action(
                "No duplicates found",
                "No duplicate rows detected in dataset"
            )

        return self

    def clean_text_columns(self):
        """Clean and standardize text columns"""

        text_cols = self.df.select_dtypes(include=['object']).columns

        for col in text_cols:
            if col in ['name', 'email', 'phone']:
                # Strip whitespace
                self.df[col] = self.df[col].str.strip()

                # Handle text columns
                if col == 'name':
                    # Capitalize names properly
                    self.df[col] = self.df[col].str.title()
                    # Replace N/A, Unknown, etc.
                    self.df.loc[self.df[col].isin(['N/A', 'Unknown', 'Null', 'None']), col] = np.nan

                elif col == 'email':
                    # Validate email format (basic)
                    email_pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
                    invalid_emails = ~self.df[col].str.match(email_pattern, na=False)
                    self.df.loc[invalid_emails, col] = np.nan

                elif col == 'phone':
                    # Standardize phone numbers
                    self.df[col] = self.df[col].str.replace(r'\D', '', regex=True)  # Keep only digits
                    self.df.loc[self.df[col].str.len() < 10, col] = np.nan  # Invalid phone numbers

                    # Format phone numbers (XXX-XXX-XXXX)
                    valid_phones = self.df[col].notna()
                    self.df.loc[valid_phones, col] = self.df.loc[valid_phones, col].apply(
                        lambda x: f"{x[:3]}-{x[3:6]}-{x[6:10]}" if len(str(x)) == 10 else x
                    )

                self.log_action(
                    "Cleaned text columns",
                    f"Column '{col}': standardized formatting and removed invalid entries"
                )

        return self

    def handle_outliers(self, column, method='iqr', threshold=1.5):
        """Detect and handle outliers in numeric columns"""

        if column not in self.df.columns:
            return self

        before_count = len(self.df)

        if method == 'iqr':
            Q1 = self.df[column].quantile(0.25)
            Q3 = self.df[column].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - threshold * IQR
            upper_bound = Q3 + threshold * IQR

            outliers = self.df[(self.df[column] < lower_bound) | (self.df[column] > upper_bound)]
            outlier_count = len(outliers)

            # Cap outliers instead of removing
            self.df[column] = self.df[column].clip(lower_bound, upper_bound)

            self.log_action(
                "Handled outliers",
                f"Column '{column}': capped {outlier_count} outliers (bounds: {lower_bound:.2f} - {upper_bound:.2f})"
            )

        return self

    def validate_data(self):
        """Perform final data validation"""

        validation_results = []

        # Check for missing values
        missing_cols = self.df.columns[self.df.isnull().any()].tolist()
        if missing_cols:
            validation_results.append(f"⚠️ Warning: Missing values still exist in: {missing_cols}")
        else:
            validation_results.append("✅ No missing values found")

        # Check data types
        expected_types = {
            'customer_id': 'int64',
            'age': 'float64',
            'total_purchases': 'float64',
            'registration_date': 'datetime64[ns]'
        }

        for col, expected in expected_types.items():
            if col in self.df.columns:
                if self.df[col].dtype != expected:
                    validation_results.append(f"⚠️ Column '{col}' has type {self.df[col].dtype}, expected {expected}")
                else:
                    validation_results.append(f"✅ Column '{col}' type is correct")

        # Check for duplicates
        duplicates = self.df.duplicated().sum()
        if duplicates > 0:
            validation_results.append(f"⚠️ Warning: {duplicates} duplicate rows found")
        else:
            validation_results.append("✅ No duplicate rows")

        # Check for data range validity
        if 'age' in self.df.columns:
            invalid_ages = ((self.df['age'] < 0) | (self.df['age'] > 120)).sum()
            if invalid_ages > 0:
                validation_results.append(f"⚠️ Warning: {invalid_ages} rows have invalid age values")
            else:
                validation_results.append("✅ Age values are within valid range")

        if 'total_purchases' in self.df.columns:
            negative_purchases = (self.df['total_purchases'] < 0).sum()
            if negative_purchases > 0:
                validation_results.append(f"⚠️ Warning: {negative_purchases} rows have negative purchases")
            else:
                validation_results.append("✅ Purchase amounts are valid")

        # Log validation results
        for result in validation_results:
            self.log_action("Validation", result)

        return validation_results

    def get_cleaning_summary(self):
        """Get comprehensive cleaning summary"""

        summary = {
            'original_shape': self.initial_shape,
            'cleaned_shape': self.df.shape,
            'rows_removed': self.initial_shape[0] - self.df.shape[0],
            'columns_cleaned': len(self.df.columns),
            'missing_values_handled': self.df.isnull().sum().sum(),
            'memory_usage_mb': self.df.memory_usage(deep=True).sum() / 1024 / 1024
        }

        return summary

    def save_cleaned_data(self, output_path):
        """Save cleaned dataset"""
        self.df.to_csv(output_path, index=False)
        self.log_action("Saved cleaned data", f"Saved to {output_path}")

    def save_cleaning_log(self, log_path):
        """Save cleaning log to file"""
        log_df = pd.DataFrame(self.cleaning_log)
        log_df.to_csv(log_path, index=False)

        # Also save as text file for readability
        with open(log_path.replace('.csv', '.txt'), 'w') as f:
            f.write("="*80 + "\n")
            f.write(f"CLEANING LOG: {self.name}\n")
            f.write(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write("="*80 + "\n\n")

            for entry in self.cleaning_log:
                f.write(f"[{entry['timestamp']}] {entry['action']}\n")
                f.write(f"  Details: {entry['details']}\n")
                if entry['before'] and entry['after']:
                    f.write(f"  Count: {entry['before']} → {entry['after']}\n")
                f.write("\n")

        self.log_action("Saved cleaning log", f"Saved to {log_path}")

<>:33: SyntaxWarning: invalid escape sequence '\w'
<>:33: SyntaxWarning: invalid escape sequence '\w'
/tmp/ipykernel_3245/3474457774.py:33: SyntaxWarning: invalid escape sequence '\w'
  .str.replace('[^\w]', '_', regex=True)  # Replace special chars


In [41]:
class DataCleaner:
    """Complete data cleaning utility class"""

    def __init__(self, df, name="dataset"):
        self.df = df.copy()
        self.name = name
        self.cleaning_log = []
        self.initial_shape = df.shape

    def log_action(self, action, details, before_count=None, after_count=None):
        """Log cleaning actions"""
        log_entry = {
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'action': action,
            'details': details,
            'before': before_count,
            'after': after_count
        }
        self.cleaning_log.append(log_entry)
        print(f"✓ {action}: {details}")

    def standardize_column_names(self):
        """Clean and standardize column names"""
        original_cols = self.df.columns.tolist()

        # Column name cleaning rules
        self.df.columns = (
            self.df.columns
            .str.lower()  # Convert to lowercase
            .str.strip()  # Remove leading/trailing spaces
            .str.replace(' ', '_')  # Replace spaces with underscores
            .str.replace('-', '_')  # Replace hyphens with underscores
            .str.replace('[^\w]', '_', regex=True)  # Replace special chars
            .str.replace('__+', '_', regex=True)  # Remove duplicate underscores
        )

        self.log_action(
            "Standardized column names",
            f"Renamed: {original_cols} → {self.df.columns.tolist()}"
        )
        return self

    def fix_data_types(self):
        """Detect and fix incorrect data types"""

        # 1. Fix numeric columns
        numeric_columns = ['age', 'total_purchases']
        for col in numeric_columns:
            if col in self.df.columns:
                before_invalid = self.df[col].isna().sum()

                # Convert to numeric, coerce errors to NaN
                self.df[col] = pd.to_numeric(self.df[col], errors='coerce')

                # Handle negative values in age
                if col == 'age':
                    invalid_neg = (self.df[col] < 0).sum()
                    self.df.loc[self.df[col] < 0, col] = np.nan
                    self.df.loc[self.df[col] > 120, col] = np.nan  # Cap age at 120

                # Handle negative purchases
                if col == 'total_purchases':
                    self.df.loc[self.df[col] < 0, col] = np.nan

                after_invalid = self.df[col].isna().sum()
                self.log_action(
                    "Fixed data types",
                    f"Column '{col}': converted {before_invalid} invalid values to NaN"
                )

        # 2. Fix dates
        if 'registration_date' in self.df.columns:
            before_invalid = self.df['registration_date'].isna().sum()

            # Try multiple date formats
            date_formats = ['%Y-%m-%d', '%Y/%m/%d', '%d-%m-%Y', '%Y.%m.%d', '%m/%d/%Y']

            for date_format in date_formats:
                mask = self.df['registration_date'].notna()
                self.df.loc[mask, 'registration_date'] = pd.to_datetime(
                    self.df.loc[mask, 'registration_date'],
                    format=date_format,
                    errors='ignore'
                )

            # Final conversion
            self.df['registration_date'] = pd.to_datetime(
                self.df['registration_date'],
                errors='coerce'
            )

            after_invalid = self.df['registration_date'].isna().sum()
            self.log_action(
                "Fixed date formats",
                f"Column 'registration_date': fixed {before_invalid} invalid dates"
            )

        # 3. Clean categorical columns
        if 'membership_status' in self.df.columns:
            valid_statuses = ['gold', 'silver', 'bronze', 'platinum']
            before_invalid = (~self.df['membership_status'].str.lower().isin(valid_statuses)).sum()

            self.df['membership_status'] = (
                self.df['membership_status']
                .str.lower()
                .str.strip()
            )
            self.df.loc[~self.df['membership_status'].isin(valid_statuses), 'membership_status'] = np.nan

            self.log_action(
                "Cleaned categorical data",
                f"Column 'membership_status': standardized {before_invalid} invalid entries"
            )

        return self

    def handle_missing_values(self, strategy='auto'):
        """Handle missing values with various strategies"""

        missing_before = self.df.isnull().sum().sum()

        # For numeric columns
        numeric_cols = self.df.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            if self.df[col].isnull().sum() > 0:
                if strategy == 'auto':
                    # Use median for skewed data, mean for normal
                    skewness = self.df[col].skew() if len(self.df[col].dropna()) > 1 else 0
                    if abs(skewness) > 1:
                        fill_value = self.df[col].median()
                        method = 'median'
                    else:
                        fill_value = self.df[col].mean()
                        method = 'mean'
                elif strategy == 'mean':
                    fill_value = self.df[col].mean()
                    method = 'mean'
                elif strategy == 'median':
                    fill_value = self.df[col].median()
                    method = 'median'
                elif strategy == 'mode':
                    fill_value = self.df[col].mode().iloc[0] if not self.df[col].mode().empty else 0
                    method = 'mode'
                else:
                    fill_value = 0
                    method = 'zero'

                self.df[col].fillna(fill_value, inplace=True)
                self.log_action(
                    "Handled missing values",
                    f"Column '{col}': filled {self.df[col].isnull().sum()} missing values with {method} ({fill_value:.2f})"
                )

        # For categorical columns
        categorical_cols = self.df.select_dtypes(include=['object']).columns
        for col in categorical_cols:
            if self.df[col].isnull().sum() > 0:
                mode_value = self.df[col].mode().iloc[0] if not self.df[col].mode().empty else 'unknown'
                self.df[col].fillna(mode_value, inplace=True)
                self.log_action(
                    "Handled missing values",
                    f"Column '{col}': filled missing values with mode '{mode_value}'"
                )

        missing_after = self.df.isnull().sum().sum()
        self.log_action(
            "Missing values summary",
            f"Reduced missing values from {missing_before} to {missing_after}",
            missing_before,
            missing_after
        )

        return self

    def remove_duplicates(self, subset=None, keep='first'):
        """Remove duplicate rows"""
        before_count = len(self.df)

        if subset is None:
            # Check all columns
            self.df = self.df.drop_duplicates(keep=keep)
            duplicate_cols = "all columns"
        else:
            self.df = self.df.drop_duplicates(subset=subset, keep=keep)
            duplicate_cols = f"columns: {subset}"

        after_count = len(self.df)
        duplicates_removed = before_count - after_count

        if duplicates_removed > 0:
            self.log_action(
                "Removed duplicates",
                f"Removed {duplicates_removed} duplicate rows based on {duplicate_cols}",
                before_count,
                after_count
            )
        else:
            self.log_action(
                "No duplicates found",
                "No duplicate rows detected in dataset"
            )

        return self

    def clean_text_columns(self):
        """Clean and standardize text columns"""

        text_cols = self.df.select_dtypes(include=['object']).columns

        for col in text_cols:
            if col in ['name', 'email', 'phone']:
                # Strip whitespace
                self.df[col] = self.df[col].str.strip()

                # Handle text columns
                if col == 'name':
                    # Capitalize names properly
                    self.df[col] = self.df[col].str.title()
                    # Replace N/A, Unknown, etc.
                    self.df.loc[self.df[col].isin(['N/A', 'Unknown', 'Null', 'None']), col] = np.nan

                elif col == 'email':
                    # Validate email format (basic)
                    email_pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
                    invalid_emails = ~self.df[col].str.match(email_pattern, na=False)
                    self.df.loc[invalid_emails, col] = np.nan

                elif col == 'phone':
                    # Standardize phone numbers
                    self.df[col] = self.df[col].str.replace(r'\D', '', regex=True)  # Keep only digits
                    self.df.loc[self.df[col].str.len() < 10, col] = np.nan  # Invalid phone numbers

                    # Format phone numbers (XXX-XXX-XXXX)
                    valid_phones = self.df[col].notna()
                    self.df.loc[valid_phones, col] = self.df.loc[valid_phones, col].apply(
                        lambda x: f"{x[:3]}-{x[3:6]}-{x[6:10]}" if len(str(x)) == 10 else x
                    )

                self.log_action(
                    "Cleaned text columns",
                    f"Column '{col}': standardized formatting and removed invalid entries"
                )

        return self

    def handle_outliers(self, column, method='iqr', threshold=1.5):
        """Detect and handle outliers in numeric columns"""

        if column not in self.df.columns:
            return self

        before_count = len(self.df)

        if method == 'iqr':
            Q1 = self.df[column].quantile(0.25)
            Q3 = self.df[column].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - threshold * IQR
            upper_bound = Q3 + threshold * IQR

            outliers = self.df[(self.df[column] < lower_bound) | (self.df[column] > upper_bound)]
            outlier_count = len(outliers)

            # Cap outliers instead of removing
            self.df[column] = self.df[column].clip(lower_bound, upper_bound)

            self.log_action(
                "Handled outliers",
                f"Column '{column}': capped {outlier_count} outliers (bounds: {lower_bound:.2f} - {upper_bound:.2f})"
            )

        return self

    def validate_data(self):
        """Perform final data validation"""

        validation_results = []

        # Check for missing values
        missing_cols = self.df.columns[self.df.isnull().any()].tolist()
        if missing_cols:
            validation_results.append(f"⚠️ Warning: Missing values still exist in: {missing_cols}")
        else:
            validation_results.append("✅ No missing values found")

        # Check data types
        expected_types = {
            'customer_id': 'int64',
            'age': 'float64',
            'total_purchases': 'float64',
            'registration_date': 'datetime64[ns]'
        }

        for col, expected in expected_types.items():
            if col in self.df.columns:
                if self.df[col].dtype != expected:
                    validation_results.append(f"⚠️ Column '{col}' has type {self.df[col].dtype}, expected {expected}")
                else:
                    validation_results.append(f"✅ Column '{col}' type is correct")

        # Check for duplicates
        duplicates = self.df.duplicated().sum()
        if duplicates > 0:
            validation_results.append(f"⚠️ Warning: {duplicates} duplicate rows found")
        else:
            validation_results.append("✅ No duplicate rows")

        # Check for data range validity
        if 'age' in self.df.columns:
            invalid_ages = ((self.df['age'] < 0) | (self.df['age'] > 120)).sum()
            if invalid_ages > 0:
                validation_results.append(f"⚠️ Warning: {invalid_ages} rows have invalid age values")
            else:
                validation_results.append("✅ Age values are within valid range")

        if 'total_purchases' in self.df.columns:
            negative_purchases = (self.df['total_purchases'] < 0).sum()
            if negative_purchases > 0:
                validation_results.append(f"⚠️ Warning: {negative_purchases} rows have negative purchases")
            else:
                validation_results.append("✅ Purchase amounts are valid")

        # Log validation results
        for result in validation_results:
            self.log_action("Validation", result)

        return validation_results

    def get_cleaning_summary(self):
        """Get comprehensive cleaning summary"""

        summary = {
            'original_shape': self.initial_shape,
            'cleaned_shape': self.df.shape,
            'rows_removed': self.initial_shape[0] - self.df.shape[0],
            'columns_cleaned': len(self.df.columns),
            'missing_values_handled': self.df.isnull().sum().sum(),
            'memory_usage_mb': self.df.memory_usage(deep=True).sum() / 1024 / 1024
        }

        return summary

    def save_cleaned_data(self, output_path):
        """Save cleaned dataset"""
        self.df.to_csv(output_path, index=False)
        self.log_action("Saved cleaned data", f"Saved to {output_path}")

    def save_cleaning_log(self, log_path):
        """Save cleaning log to file"""
        log_df = pd.DataFrame(self.cleaning_log)
        log_df.to_csv(log_path, index=False)

        # Also save as text file for readability
        with open(log_path.replace('.csv', '.txt'), 'w') as f:
            f.write("="*80 + "\n")
            f.write(f"CLEANING LOG: {self.name}\n")
            f.write(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write("="*80 + "\n\n")

            for entry in self.cleaning_log:
                f.write(f"[{entry['timestamp']}] {entry['action']}\n")
                f.write(f"  Details: {entry['details']}\n")
                if entry['before'] and entry['after']:
                    f.write(f"  Count: {entry['before']} → {entry['after']}\n")
                f.write("\n")

        self.log_action("Saved cleaning log", f"Saved to {log_path}")

<>:33: SyntaxWarning: invalid escape sequence '\w'
<>:33: SyntaxWarning: invalid escape sequence '\w'
/tmp/ipykernel_3245/3474457774.py:33: SyntaxWarning: invalid escape sequence '\w'
  .str.replace('[^\w]', '_', regex=True)  # Replace special chars


In [43]:
# Load messy data
print("\n" + "="*60)
print("STARTING DATA CLEANING PIPELINE")
print("="*60)

df_messy = pd.read_csv('data/raw/messy_customers.csv')

# Initialize cleaner
cleaner = DataCleaner(df_messy, name="Customer Data")

# Run cleaning pipeline
print("\n🔧 Step 1: Standardizing Column Names")
cleaner.standardize_column_names()

print("\n🔧 Step 2: Fixing Data Types")
cleaner.fix_data_types()

print("\n🔧 Step 3: Cleaning Text Columns")
cleaner.clean_text_columns()

print("\n🔧 Step 4: Handling Missing Values")
cleaner.handle_missing_values(strategy='auto')

print("\n🔧 Step 5: Removing Duplicates")
cleaner.remove_duplicates(subset=['customer_id', 'name'], keep='first')

print("\n🔧 Step 6: Handling Outliers")
if 'total_purchases' in cleaner.df.columns:
    cleaner.handle_outliers('total_purchases', method='iqr')

print("\n🔧 Step 7: Final Validation")
validation_results = cleaner.validate_data()

print("\n" + "="*60)
print("CLEANING COMPLETE")
print("="*60)

# Display summary
summary = cleaner.get_cleaning_summary()
print("\n📊 Cleaning Summary:")
for key, value in summary.items():
    print(f"   {key}: {value}")

# Show validation results
print("\n✅ Validation Results:")
for result in validation_results:
    print(f"   {result}")


STARTING DATA CLEANING PIPELINE

🔧 Step 1: Standardizing Column Names
✓ Standardized column names: Renamed: ['customer id', 'Name', 'Email', 'Phone', 'Age', 'Registration_Date', 'Total_Purchases', 'Membership_Status', 'Last_Active'] → ['customer_id', 'name', 'email', 'phone', 'age', 'registration_date', 'total_purchases', 'membership_status', 'last_active']

🔧 Step 2: Fixing Data Types
✓ Fixed data types: Column 'age': converted 0 invalid values to NaN
✓ Fixed data types: Column 'total_purchases': converted 2 invalid values to NaN
✓ Fixed date formats: Column 'registration_date': fixed 0 invalid dates
✓ Cleaned categorical data: Column 'membership_status': standardized 3 invalid entries

🔧 Step 3: Cleaning Text Columns
✓ Cleaned text columns: Column 'name': standardized formatting and removed invalid entries
✓ Cleaned text columns: Column 'email': standardized formatting and removed invalid entries
✓ Cleaned text columns: Column 'phone': standardized formatting and removed invalid ent

/tmp/ipykernel_3245/3474457774.py:80: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  self.df.loc[mask, 'registration_date'] = pd.to_datetime(
/tmp/ipykernel_3245/3474457774.py:80: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  self.df.loc[mask, 'registration_date'] = pd.to_datetime(
/tmp/ipykernel_3245/3474457774.py:80: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  self.df.loc[mask, 'registration_date'] = pd.to_datetime(
/tmp/ipykernel_3245/3474457774.py:80: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  self.df.loc[mask, 're

In [44]:
print("\n" + "="*60)
print("BEFORE vs AFTER COMPARISON")
print("="*60)

# Display cleaned data
print("\n✨ CLEANED DATASET (First 10 rows):")
print(cleaner.df.head(10))

print("\n📋 Data Types After Cleaning:")
print(cleaner.df.dtypes)

print("\n📊 Statistical Summary After Cleaning:")
print(cleaner.df.describe())

print("\n🔍 Missing Values After Cleaning:")
print(cleaner.df.isnull().sum())

# Compare with original
print("\n" + "="*60)
print("IMPROVEMENT METRICS")
print("="*60)

original_missing = df_messy.isnull().sum().sum()
cleaned_missing = cleaner.df.isnull().sum().sum()
missing_reduction = ((original_missing - cleaned_missing) / original_missing * 100) if original_missing > 0 else 0

original_duplicates = df_messy.duplicated().sum()
cleaned_duplicates = cleaner.df.duplicated().sum()

print(f"Missing values reduced: {original_missing} → {cleaned_missing} ({missing_reduction:.1f}% reduction)")
print(f"Duplicates removed: {original_duplicates} → {cleaned_duplicates}")
print(f"Rows: {len(df_messy)} → {len(cleaner.df)}")
print(f"Columns standardized: {len(df_messy.columns)} → {len(cleaner.df.columns)}")


BEFORE vs AFTER COMPARISON

✨ CLEANED DATASET (First 10 rows):
   customer_id            name                email         phone        age  \
0            1      John Smith       john@email.com  123-456-7890  25.000000   
1            2        Jane Doe  jane.doe@domain.com  987-654-3210  31.090909   
2            3     Bob Johnson    charlie@email.com  444-555-6666  31.090909   
3            4     Alice Brown    charlie@email.com  444-555-6666  30.000000   
4            5  Charlie Wilson    charlie@email.com  444-555-6666  42.000000   
5            6      Emma Davis       emma@email.com  999-888-7777  18.000000   
6            7     Michael Lee    charlie@email.com  111-222-3333  31.090909   
7            8     Sara Parker       sara@email.com  444-555-6666  35.000000   
8            9       David Kim      david@email.com  777-888-9999  27.000000   
9           10       Lisa Wong       lisa@email.com  000-111-2222  31.090909   

  registration_date  total_purchases membership_status 

In [45]:
print("\n" + "="*60)
print("SAVING RESULTS")
print("="*60)

# Save cleaned data
cleaner.save_cleaned_data('data/cleaned/cleaned_customers.csv')

# Save cleaning log
cleaner.save_cleaning_log('data/cleaned/cleaning_log.csv')

# Generate HTML report
def generate_html_report(cleaner):
    """Generate an HTML report of the cleaning process"""

    html_content = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <title>Data Cleaning Report - {cleaner.name}</title>
        <style>
            body {{ font-family: Arial, sans-serif; margin: 20px; background: #f5f5f5; }}
            .container {{ max-width: 1200px; margin: auto; background: white; padding: 20px; border-radius: 10px; }}
            h1 {{ color: #2c3e50; }}
            h2 {{ color: #34495e; border-bottom: 2px solid #3498db; padding-bottom: 5px; }}
            .summary {{ background: #ecf0f1; padding: 15px; border-radius: 5px; margin: 20px 0; }}
            .success {{ color: #27ae60; }}
            .warning {{ color: #e67e22; }}
            .error {{ color: #e74c3c; }}
            table {{ width: 100%; border-collapse: collapse; margin: 10px 0; }}
            th, td {{ border: 1px solid #ddd; padding: 8px; text-align: left; }}
            th {{ background-color: #3498db; color: white; }}
            tr:nth-child(even) {{ background-color: #f2f2f2; }}
            .log-entry {{ background: #f8f9fa; margin: 10px 0; padding: 10px; border-left: 4px solid #3498db; }}
            .timestamp {{ color: #7f8c8d; font-size: 0.9em; }}
        </style>
    </head>
    <body>
        <div class="container">
            <h1>🧹 Data Cleaning Report: {cleaner.name}</h1>
            <p>Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>

            <div class="summary">
                <h2>📊 Summary Statistics</h2>
                <table>
                    <tr><th>Metric</th><th>Value</th></tr>
                    <tr><td>Original shape</td><td>{cleaner.initial_shape}</td></tr>
                    <tr><td>Cleaned shape</td><td>{cleaner.df.shape}</td></tr>
                    <tr><td>Rows removed</td><td>{cleaner.initial_shape[0] - cleaner.df.shape[0]}</td></tr>
                    <tr><td>Missing values after cleaning</td><td>{cleaner.df.isnull().sum().sum()}</td></tr>
                    <tr><td>Memory usage</td><td>{cleaner.df.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB</td></tr>
                </table>
            </div>

            <h2>📋 Cleaning Log</h2>
    """

    for entry in cleaner.cleaning_log:
        html_content += f"""
            <div class="log-entry">
                <div class="timestamp">[{entry['timestamp']}]</div>
                <strong>{entry['action']}</strong><br>
                {entry['details']}
        """
        if entry['before'] and entry['after']:
            html_content += f"<br><span class='success'>Count: {entry['before']} → {entry['after']}</span>"
        html_content += "</div>"

    html_content += f"""
            <h2>📈 Cleaned Data Preview</h2>
            {cleaner.df.head(10).to_html()}

            <h2>🔍 Data Types After Cleaning</h2>
            {pd.DataFrame(cleaner.df.dtypes).to_html()}
        </div>
    </body>
    </html>
    """

    with open('outputs/cleaning_report.html', 'w') as f:
        f.write(html_content)

    print("✓ HTML report generated: outputs/cleaning_report.html")

generate_html_report(cleaner)

print("\n✅ All outputs saved successfully!")
print("   - Cleaned data: data/cleaned/cleaned_customers.csv")
print("   - Cleaning log: data/cleaned/cleaning_log.csv")
print("   - HTML report: outputs/cleaning_report.html")


SAVING RESULTS
✓ Saved cleaned data: Saved to data/cleaned/cleaned_customers.csv
✓ Saved cleaning log: Saved to data/cleaned/cleaning_log.csv
✓ HTML report generated: outputs/cleaning_report.html

✅ All outputs saved successfully!
   - Cleaned data: data/cleaned/cleaned_customers.csv
   - Cleaning log: data/cleaned/cleaning_log.csv
   - HTML report: outputs/cleaning_report.html


In [46]:
# Create standalone functions for quick cleaning

def quick_clean_dataframe(df, verbose=True):
    """Quick cleaning for any dataframe"""

    cleaner = DataCleaner(df, name="Quick_Clean")

    cleaner.standardize_column_names()
    cleaner.fix_data_types()
    cleaner.handle_missing_values(strategy='auto')
    cleaner.remove_duplicates()
    cleaner.clean_text_columns()

    if verbose:
        summary = cleaner.get_cleaning_summary()
        print("\nQuick cleaning completed:")
        for key, value in summary.items():
            print(f"  {key}: {value}")

    return cleaner.df, cleaner.cleaning_log

# Example usage
def batch_clean_files(file_paths, output_dir='data/cleaned/'):
    """Clean multiple CSV files"""

    for file_path in file_paths:
        print(f"\nCleaning {file_path}...")
        df = pd.read_csv(file_path)
        cleaned_df, log = quick_clean_dataframe(df, verbose=False)

        output_path = f"{output_dir}/cleaned_{os.path.basename(file_path)}"
        cleaned_df.to_csv(output_path, index=False)
        print(f"  ✓ Saved to {output_path}")

print("\n📚 Reusable functions created:")
print("   - quick_clean_dataframe(): Clean any dataframe")
print("   - batch_clean_files(): Clean multiple CSV files")


📚 Reusable functions created:
   - quick_clean_dataframe(): Clean any dataframe
   - batch_clean_files(): Clean multiple CSV files
